# 05 · XGBoost

**Goal (Phase 5):** see whether gradient boosting beats linear regression **out of sample**.

- **The bar:** linear regression test MAE **4.50** on 2025-26 (walk-forward average 4.50).
- **Start calm:** shallow trees, low learning rate, high `min_child_weight`, and **early stopping** on a validation season.
- **Three-way time split:** train 2021-22 to 2023-24, **validation 2024-25** (only for early stopping and tuning), test 2025-26 (touched once at the end).
- **Watch the train vs test gap.** Model 1's boosting scored 0.43 on train and 1.24 on test.
- Keep XGBoost only if it **clearly** beats 4.50.

In [11]:
import pandas as pd
import numpy as np
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from fbml.config import ROOT

pd.set_option("display.max_columns", 50)

PROCESSED = ROOT / "nba" / "data" / "processed"
features = pd.read_parquet(PROCESSED / "features.parquet")

ID_COLS = ['season', 'player_id', 'player', 'team', 'opponent', 'game_id', 'date']
TARGET = 'points'
FEATURES = [c for c in features.columns if c not in ID_COLS + [TARGET, 'minutes']]
LIN_REG_TEST_MAE = 4.50  # the bar from Phase 4

model_df = features.dropna(subset=FEATURES)
print(model_df.shape, "|", len(FEATURES), "features | xgboost", xgb.__version__)

(119631, 31) | 22 features | xgboost 3.4.1


In [ ]:
train_df = model_df[model_df.season < 22024]
valid_df = model_df[model_df.season == 22024]
test_df = model_df[model_df.season == 22025]
print(train_df.shape, valid_df.shape, test_df.shape)
train_df['date'].min()



(69720, 31) (24790, 31) (25121, 31)


Timestamp('2021-11-08 00:00:00')

In [12]:
# calm boosting: small trees, small steps, big leaves; early stopping watches the valid season
xgb_model = XGBRegressor(
    n_estimators=2000,
    max_depth=3,
    learning_rate=0.03,
    min_child_weight=50,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=100,
    eval_metric="mae",
    random_state=42,
)

xgb_model.fit(
    train_df[FEATURES],
    train_df[TARGET],
    eval_set=[(valid_df[FEATURES], valid_df[TARGET])],
    verbose=200,
)

train_mae = mean_absolute_error(train_df[TARGET], xgb_model.predict(train_df[FEATURES]))
valid_mae = mean_absolute_error(valid_df[TARGET], xgb_model.predict(valid_df[FEATURES]))
print(f"trees kept: {xgb_model.best_iteration + 1}")
print(f"MAE  train: {train_mae:.3f}  valid: {valid_mae:.3f}  (linear regression on this valid season: 4.53)")

[0]	validation_0-mae:6.96046
[200]	validation_0-mae:4.53129
[400]	validation_0-mae:4.52099
[531]	validation_0-mae:4.52126
trees kept: 432
MAE  train: 4.487  valid: 4.520  (linear regression on this valid season: 4.53)
